## Thêm các thư viện cần thiết

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from autoviz.AutoViz_Class import AutoViz_Class

In [ ]:
data_path = r'D:\DOAN\machine_learning\lab05\data\marketing_campaign.csv'
report_dir = r'D:\DOAN\machine_learning\lab05\Lab1_EDA\reports\autoviz_output'
os.makedirs(report_dir, exist_ok=True)

In [ ]:
df=pd.read_csv(data_path, sep='\t')

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

* Cùng tập dữ liệu Marketing Campaign với bài SweetViz (2240 dòng, 29 cột), đọc với sep='\t'.

* AutoViz tự chọn loại biểu đồ phù hợp cho từng biến, tự cảnh báo các vấn đề của dữ liệu (thiếu, ngoại lai, lệch, cột hằng số, tương quan cao) nên chỉ cần truyền dữ liệu vào.

## AutoViz

AutoViz là thư viện EDA tự động, chỉ cần một dòng code AV.AutoViz(...) là vẽ ra hàng loạt biểu đồ. Các tính năng chính:

* Tự động nhận dạng kiểu biến và chọn biểu đồ phù hợp: histogram, boxplot, violin, scatter, heatmap tương quan, pair plot...

* Tự phát hiện vấn đề chất lượng dữ liệu: cột hằng số, tỷ lệ thiếu, cột có độ tương quan cao, ngoại lai, phân bố lệch.

* **depVar**: chọn biến mục tiêu để vẽ biểu đồ theo từng lớp của biến đó.

* **chart_format**: định dạng xuất biểu đồ (png, svg, html...), kết hợp **save_plot_dir** để lưu ra thư mục.

* Có thể đọc từ file (filename) hoặc từ DataFrame có sẵn (dfte).

Cài đặt: pip install autoviz

### 1. Chạy AutoViz trực tiếp trên file dữ liệu

In [ ]:
AV = AutoViz_Class()

dft = AV.AutoViz(
    filename=data_path,
    sep='\t',
    depVar='',
    dfte=None,
    header=0,
    verbose=1,
    lowess=False,
    chart_format='png',
    max_rows_analyzed=150000,
    max_cols_analyzed=30,
    save_plot_dir=os.path.join(report_dir, 'all')
)

### 2. Chạy sau khi tiền xử lý và chọn biến mục tiêu Response

In [ ]:
df_clean = df.drop(columns=['ID', 'Z_CostContact', 'Z_Revenue']).copy()
df_clean['Dt_Customer'] = pd.to_datetime(df_clean['Dt_Customer'], format='%d-%m-%Y')
df_clean['Age'] = 2014 - df_clean['Year_Birth']
df_clean = df_clean.drop(columns=['Year_Birth', 'Dt_Customer'])

df_clean.shape

In [ ]:
dft_target = AV.AutoViz(
    filename='',
    sep=',',
    depVar='Response',
    dfte=df_clean,
    header=0,
    verbose=1,
    lowess=False,
    chart_format='png',
    max_rows_analyzed=150000,
    max_cols_analyzed=30,
    save_plot_dir=os.path.join(report_dir, 'response')
)

## Nhận xét

* Lần chạy đầu AutoViz đưa ra cảnh báo về các cột không có giá trị phân tích (Z_CostContact, Z_Revenue chỉ có 1 giá trị, ID là định danh) và cột Income bị thiếu một ít, nên ở lần chạy sau đã bỏ các cột này, đổi Year_Birth thành Age (tính theo năm 2014, thời điểm thu thập dữ liệu) và bỏ Dt_Customer (kiểu ngày).

* Các biến chi tiêu (Mnt...) và Income lệch phải, nhiều ngoại lai. Ma trận tương quan cho thấy nhóm biến chi tiêu và số lần mua (NumCatalogPurchases, NumStorePurchases) tương quan dương với nhau và với Income, còn Kidhome tương quan nghịch.

* Khi đặt depVar='Response', biểu đồ được chia theo khách hàng có và không chấp nhận chiến dịch, giúp thấy biến nào phân biệt được hai nhóm. Lớp Response = 1 chiếm tỷ lệ nhỏ nên dữ liệu mất cân bằng, cần lưu ý nếu xây mô hình phân loại.

* So với SweetViz: AutoViz vẽ nhiều dạng biểu đồ hơn (scatter, pair plot, violin) và lưu trực tiếp thành ảnh, còn SweetViz cho báo cáo HTML gọn và có so sánh hai nhóm. Hai công cụ bổ sung cho nhau.

* Hạn chế: ít tuỳ chỉnh biểu đồ, sinh nhiều ảnh nên chậm hơn, và cũng có thể lỗi tương thích với phiên bản pandas/numpy mới.